# ۴. ارزیابی و گزارش پژوهشی

این مرحله مدل را آموزش نمی‌دهد. خروجی اجرای واقعی مدل پایه و در صورت وجود وزن آموزش‌دیده را تجمیع می‌کند.
اگر تنها وزن آماده را اجرا کرده‌اید، همان یک سیستم را گزارش کنید؛ نتیجهٔ آموزش‌دیده را حدس نزنید.
ابتدا مرحلهٔ 12 را برای هر وزن روی **همان test ثابت** اجرا کنید: `MANIFEST_SPLIT="test"` و تعداد نمونهٔ یکسان.
خروجی validation یا اجرای یک‌نمونه‌ای را به‌جای نتیجهٔ کامل test گزارش نکنید.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/manzourali/English-Streaming-ASR.git"
REPOSITORY_REF = "master"
ROOT = Path("/kaggle/working/English-Streaming-ASR")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(ROOT)], check=True)
os.chdir(ROOT)
if not (ROOT / "scripts/check_surt_assets.py").is_file():
    raise RuntimeError("This GitHub checkout predates the Kaggle workflow update. Publish the updated source, then start a fresh session.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[kaggle,dev]"], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
ARTIFACTS = Path("/kaggle/working/asr_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str, arguments)], check=True)

In [ ]:
import yaml
DATA_ROOT = Path("/kaggle/working/asr_artifacts/data/pilot_v1")
# Select exactly one full test-run records file per system; avoid duplicate smoke runs.
RECORD_FILES = []
if not RECORD_FILES:
    raise RuntimeError("Set RECORD_FILES to actual *_evaluation_records.jsonl files from step 12. Attach previous notebook outputs if needed.")
from streaming_asr.evaluation.records import read_evaluation_records
records = [record for path in RECORD_FILES for record in read_evaluation_records(path)]
if not records or any(record.split != "test" or record.metric != "pi_wer" for record in records):
    raise ValueError("Expected actual test PI-WER records.")
import hashlib
expected_hash = hashlib.sha256((DATA_ROOT / "manifests/test.jsonl").read_bytes()).hexdigest()
if any(r.metadata.get("manifest_sha256") != expected_hash for r in records):
    raise ValueError("Records were not produced from this exact version of the test manifest.")
seen = set()
by_system = {}
for record in records:
    key = (record.system, record.dataset, record.sample_id)
    if key in seen:
        raise ValueError(f"Repeated result: {key}. Select one run per system.")
    seen.add(key)
    by_system.setdefault(record.system, set()).add((record.dataset, record.sample_id))
if len({frozenset(items) for items in by_system.values()}) != 1:
    raise ValueError("Systems must use identical test examples and dataset IDs.")
from streaming_asr.datasets.manifests import read_overlap_manifest
expected = {(DATA_ROOT.name, r.mixture_id) for r in read_overlap_manifest(DATA_ROOT / "manifests/test.jsonl")}
if next(iter(by_system.values())) != expected:
    raise ValueError("Selected run is not the complete declared test split.")
evaluation_override = ARTIFACTS / "evaluation.yaml"
freeze = {f"{name}_manifest": str(DATA_ROOT / "manifests" / f"{name}.jsonl")
          for name in ("train", "validation", "test")}
for system, key in (("surt_base", "surt_checkpoint"), ("surt_finetuned", "finetuned_surt_checkpoint")):
    used = {r.metadata.get("checkpoint") for r in records if r.system == system}
    if len(used) > 1:
        raise ValueError("One system label refers to multiple checkpoints.")
    freeze[key] = next(iter(used)) if used else None
freeze["test_manifest_sha256"] = expected_hash
evaluation_override.write_text(yaml.safe_dump({
    "frozen_system": freeze,
    "systems": {"primary": sorted(by_system)},
    "paths": {"outputs": str(ARTIFACTS / "final_evaluation")}
}, sort_keys=False))
run_script("evaluate.py", "--override", evaluation_override, "--records", *RECORD_FILES)
run_script("benchmark.py", "--records", *RECORD_FILES)

## معنی عددها و محدودیت‌ها

PI-WER این پروژه برای مخلوط‌های کوتاهِ دو منبع است. خطاها و کلمات مرجع در کل داده جمع می‌شوند.
برای مکالمهٔ طولانی با جابه‌جایی گویندگان بین کانال‌ها، این معیار کافی نیست؛ recipe رسمی LibriCSS از ORC-WER با meeteval استفاده می‌کند.
اتصال داده/خروجی زمان‌دار برای آن ارزیابی هنوز در این پروژه آماده نیست.
RTF زمان پردازش فایل تقسیم بر طول صوت است؛ به‌تنهایی تأخیر زندهٔ میکروفن یا تضمین حافظهٔ محدود را ثابت نمی‌کند.

برای تحویل پژوهشی ذخیره کنید: نسخهٔ داده و جدایی گویندگان، تنظیمات، SHA سورس/وزن، checkpoint و tokenizer، محیط GPU/CUDA، متن هر نمونه، شمارش خطا، RTF، و گزارش محدودیت‌ها.
مرحلهٔ پایان، تست native streaming روی گفتار طولانی و بررسی حافظه/تأخیر روی GPU واقعی است؛ تست‌های محلی جای آن را نمی‌گیرند.